# Alzheimer's Disease Microglia scRNA-seq Pipeline
**Ritschel Research | Computational-First Invention Screen**

**Dataset:** GSE138852 — Human prefrontal cortex snRNA-seq
80,660 nuclei, 48 individuals (24 AD, 24 controls)
Mathys et al., Nature 2019

**Goal:** Identify novel kinase inhibitors targeting
disease-associated microglia (DAM) in human AD brain
via scRNA-seq drug screening

**DAM signature:** TREM2, TYROBP, AXL, APOE, CST7,
LPL, CTSD, SPP1, GPNMB, CD9, ITGAX, CLEC7A

**Checkpoints:**
- `ad_raw.h5ad` — after data loading
- `ad_scvi.h5ad` — after scVI training
- `ad_clustered.h5ad` — after clustering
- `ad_DE_genes.csv` — after DE
- `chembl_targets_checkpoint.json` — ChEMBL mapping
- `novelty_checkpoint.json` — USPTO novelty check

**After any runtime restart:** Run Cell 1 → Cell 2 → Recovery Cell


In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install dependencies
# After install: Runtime → Restart, then Cell 2.
# ─────────────────────────────────────────────
!pip install scvi-tools scanpy anndata pandas numpy matplotlib seaborn \
             GEOparse requests tqdm scikit-learn chembl-webresource-client \
             leidenalg scipy -q

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Imports and shared utilities
# ─────────────────────────────────────────────
import os, json, time, warnings, requests, glob, gzip, shutil
import scipy.sparse as sp
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from chembl_webresource_client.new_client import new_client
warnings.filterwarnings('ignore')
sc.settings.verbosity = 1

GEO_ID   = 'GSE138852'
DATA_DIR = f'./{GEO_ID}'
BATCH_SIZE = 50
os.makedirs(DATA_DIR, exist_ok=True)

def save_checkpoint(data, path):
    with open(path, 'w') as f: json.dump(data, f)

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path, 'r') as f: data = json.load(f)
        print(f'  Checkpoint found: {len(data)} entries.')
        return data
    print('  No checkpoint found — starting fresh.')
    return {}

print('Imports successful.')

## RECOVERY CELL — Run after any runtime restart

In [ ]:
# ─────────────────────────────────────────────
# RECOVERY CELL
# ─────────────────────────────────────────────
adata = None
recovery_stage = 'none'

if os.path.exists('ad_clustered.h5ad'):
    adata = sc.read_h5ad('ad_clustered.h5ad')
    recovery_stage = 'clustered'
    print(f'Recovered from clustering: {adata.n_obs:,} cells, {adata.obs["leiden"].nunique()} clusters')
    print('  Continue from Cell 11 (marker dotplot)')
elif os.path.exists('ad_scvi.h5ad'):
    adata = sc.read_h5ad('ad_scvi.h5ad')
    recovery_stage = 'scvi'
    print(f'Recovered from scVI: {adata.n_obs:,} cells')
    print('  Continue from Cell 10 (clustering)')
elif os.path.exists('ad_raw.h5ad'):
    adata = sc.read_h5ad('ad_raw.h5ad')
    recovery_stage = 'raw'
    print(f'Recovered from raw: {adata.n_obs:,} cells')
    print('  Continue from Cell 6 (QC)')
else:
    print('No checkpoint — run from Cell 3 (download).')

print(f'\nRecovery stage: {recovery_stage}')

## Phase 1 — Download and Load Data

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Inspect GSE138852 supplementary files
# ─────────────────────────────────────────────
if os.path.exists('ad_raw.h5ad'):
    print('ad_raw.h5ad exists — skip to Recovery Cell.')
else:
    import GEOparse
    print(f'Fetching {GEO_ID} metadata...')
    gse = GEOparse.get_GEO(geo=GEO_ID, destdir=DATA_DIR, silent=True)
    print(f'Title: {gse.metadata["title"][0]}')
    print(f'Samples: {len(gse.gsms)}')
    print('\nSupplementary files:')
    for s in gse.metadata.get('supplementary_file', []):
        print(f'  {s}')
    print('\nFirst 3 samples:')
    for gsm_id, gsm in list(gse.gsms.items())[:3]:
        print(f'  {gsm_id}: {gsm.metadata["title"][0]}')
        for s in gsm.metadata.get('supplementary_file', [])[:2]:
            print(f'    {s}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Download data
# GSE138852 typically provides h5ad or sparse
# matrix files. Update filename from Cell 3.
# ─────────────────────────────────────────────
if os.path.exists('ad_raw.h5ad'):
    print('ad_raw.h5ad exists — skipping.')
else:
    BASE = f'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE138nnn/{GEO_ID}/suppl'

    # Common filename patterns for Mathys et al. 2019
    candidate_files = [
        f'{GEO_ID}_filtered_gene_bc_matrices_h5.h5.gz',
        f'{GEO_ID}_RAW.tar',
        f'{GEO_ID}_raw_counts.h5ad.gz',
        f'{GEO_ID}_all_cells.h5ad.gz',
        f'{GEO_ID}_processed.h5ad.gz',
    ]

    downloaded = []
    for fname in candidate_files:
        url = f'{BASE}/{fname}'
        out = f'{DATA_DIR}/{fname}'
        if os.path.exists(out):
            print(f'  Already exists: {fname} ({os.path.getsize(out)/1e6:.1f} MB)')
            downloaded.append(out)
            break
        try:
            r = requests.head(url, timeout=10)
            if r.status_code == 200:
                size_mb = int(r.headers.get('Content-Length', 0)) / 1e6
                print(f'  Found: {fname} ({size_mb:.0f} MB) — downloading...')
                r2 = requests.get(url, stream=True)
                with open(out, 'wb') as f:
                    for chunk in r2.iter_content(8192): f.write(chunk)
                print(f'  Done.')
                downloaded.append(out)
                break
        except: pass

    if not downloaded:
        print('Auto-download failed. Manual steps:')
        print(f'  1. Go to https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={GEO_ID}')
        print(f'  2. Download the scRNA-seq count matrix')
        print(f'  3. Upload to Colab and place in {DATA_DIR}/')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Load data
# Auto-detects h5ad, h5, or mtx format.
# ─────────────────────────────────────────────
if os.path.exists('ad_raw.h5ad'):
    print('ad_raw.h5ad exists — skipping.')
else:
    def find_and_load(data_dir):
        for f in glob.glob(f'{data_dir}/**/*.h5ad*', recursive=True):
            if f.endswith('.gz'):
                out = f[:-3]
                if not os.path.exists(out):
                    print(f'Decompressing {f}...')
                    with gzip.open(f, 'rb') as fi, open(out, 'wb') as fo:
                        shutil.copyfileobj(fi, fo)
                f = out
            print(f'Loading h5ad: {f}')
            return sc.read_h5ad(f)
        for f in glob.glob(f'{data_dir}/**/*.h5*', recursive=True):
            if not f.endswith('.h5ad'):
                if f.endswith('.gz'):
                    out = f[:-3]
                    with gzip.open(f, 'rb') as fi, open(out, 'wb') as fo:
                        shutil.copyfileobj(fi, fo)
                    f = out
                print(f'Loading h5: {f}')
                try: return sc.read_10x_h5(f)
                except: pass
        mtx_dirs = list(set(os.path.dirname(f) for f in
            glob.glob(f'{data_dir}/**/matrix.mtx*', recursive=True)))
        if mtx_dirs:
            print(f'Loading mtx: {mtx_dirs[0]}')
            return sc.read_10x_mtx(mtx_dirs[0], var_names='gene_symbols')
        raise FileNotFoundError('No data files found — check Cell 4.')

    adata = find_and_load(DATA_DIR)
    adata.var_names_make_unique()
    print(f'Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes')
    print('obs columns:', adata.obs.columns.tolist()[:15])

    # Show condition/diagnosis distribution
    for col in ['diagnosis', 'disease', 'condition', 'AD', 'Diagnosis', 'braak']:
        if col in adata.obs.columns:
            print(f'\n{col}:', adata.obs[col].value_counts().to_dict())

    adata.write('ad_raw.h5ad')
    print('Saved: ad_raw.h5ad')

## Phase 2 — QC and Preprocessing

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Load raw checkpoint if needed
# ─────────────────────────────────────────────
if 'adata' not in dir() or adata is None:
    adata = sc.read_h5ad('ad_raw.h5ad')
    print(f'Loaded: {adata.n_obs:,} cells')
    print('obs columns:', adata.obs.columns.tolist()[:15])
else:
    print(f'adata in memory: {adata.n_obs:,} cells')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — QC metrics
# snRNA-seq: expect low MT% (<5%) since nuclei
# don't contain mitochondria.
# ─────────────────────────────────────────────
if os.path.exists('ad_scvi.h5ad') or os.path.exists('ad_clustered.h5ad'):
    print('Checkpoint exists — skip.')
else:
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    print(f'Median genes/cell: {adata.obs["n_genes_by_counts"].median():.0f}')
    print(f'Median UMIs/cell:  {adata.obs["total_counts"].median():.0f}')
    print(f'Median MT%:        {adata.obs["pct_counts_mt"].median():.1f}%')
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
                 jitter=0.4, multi_panel=True)

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Filter, normalize, HVG
# snRNA-seq thresholds: lower gene min (200),
# higher MT% allowed (10%) since some MT genes
# appear in nucleus.
# ─────────────────────────────────────────────
if os.path.exists('ad_scvi.h5ad') or os.path.exists('ad_clustered.h5ad'):
    print('Checkpoint exists — skip.')
else:
    print(f'Before: {adata.n_obs:,} cells')
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=10)
    adata = adata[adata.obs['total_counts'] < 20000, :]
    adata = adata[adata.obs['pct_counts_mt'] < 10, :]
    print(f'After:  {adata.n_obs:,} cells')

    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.raw = adata

    sc.pp.highly_variable_genes(
        adata, n_top_genes=3000, subset=True,
        layer='counts', flavor='seurat_v3')
    print(f'HVGs: {adata.n_vars}')

## Phase 3 — scVI Training and Clustering

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — scVI setup and training
# GPU recommended: 80K cells ~8-10 min on T4
# ─────────────────────────────────────────────
if os.path.exists('ad_scvi.h5ad') or os.path.exists('ad_clustered.h5ad'):
    print('Checkpoint exists — skip to Recovery Cell, then Cell 10.')
else:
    batch_key = None
    for candidate in ['donor', 'sample', 'batch', 'individual', 'subject',
                       'orig.ident', 'patient', 'projid']:
        if candidate in adata.obs.columns:
            batch_key = candidate
            break
    print(f'Batch key: {batch_key}')

    scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key=batch_key)
    model = scvi.model.SCVI(adata, n_layers=2, n_latent=30, gene_likelihood='nb')

    model.train(
        max_epochs=100,
        early_stopping=True,
        early_stopping_patience=10,
        plan_kwargs={'lr': 1e-3}
    )
    adata.obsm['X_scVI'] = model.get_latent_representation()
    print('scVI complete.')

    adata.write('ad_scvi.h5ad')
    model.save('ad_scvi_model/', overwrite=True)
    print('scVI checkpoint saved.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Clustering and UMAP
# ─────────────────────────────────────────────
if os.path.exists('ad_clustered.h5ad'):
    print('Checkpoint exists — skip to Recovery Cell, then Cell 11.')
else:
    if 'X_scVI' not in adata.obsm:
        adata = sc.read_h5ad('ad_scvi.h5ad')

    sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=15)
    for res in [0.3, 0.5, 0.8]:
        sc.tl.leiden(adata, resolution=res, key_added=f'leiden_{res}')
        print(f'  Resolution {res}: {adata.obs[f"leiden_{res}"].nunique()} clusters')

    adata.obs['leiden'] = adata.obs['leiden_0.5']
    sc.tl.umap(adata)
    sc.pl.umap(adata, color='leiden', legend_loc='on data',
               title='AD prefrontal cortex clusters (Leiden 0.5)')

    # Color by diagnosis if available
    for col in ['diagnosis', 'disease', 'Diagnosis', 'AD', 'condition']:
        if col in adata.obs.columns:
            sc.pl.umap(adata, color=col, title=f'Diagnosis: {col}')
            break

    adata.write('ad_clustered.h5ad')
    print('Clustering checkpoint saved.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — Cell type marker dotplot
# Major brain cell types
# ─────────────────────────────────────────────
BRAIN_MARKERS = {
    'Microglia (DAM)': ['TREM2', 'TYROBP', 'AXL', 'CST7', 'LPL', 'GPNMB', 'CD9', 'ITGAX'],
    'Microglia (homeostatic)': ['CX3CR1', 'P2RY12', 'TMEM119', 'SALL1'],
    'Astrocytes':     ['GFAP', 'AQP4', 'SLC1A2', 'ALDH1L1'],
    'Oligodendrocytes': ['MBP', 'MOG', 'PLP1', 'MAG'],
    'OPCs':           ['PDGFRA', 'CSPG4', 'OLIG1'],
    'Excitatory neurons': ['NRGN', 'SLC17A7', 'CAMK2A'],
    'Inhibitory neurons': ['GAD1', 'GAD2', 'SLC32A1'],
    'Endothelial':    ['PECAM1', 'FLT1', 'CLDN5'],
}
all_markers = [g for genes in BRAIN_MARKERS.values() for g in genes]
present = [g for g in all_markers if g in adata.raw.var_names]
print(f'Markers present: {len(present)}/{len(all_markers)}')
if present:
    sc.pl.dotplot(adata, present, groupby='leiden',
                  title='Brain cell type markers by cluster')

## Phase 4 — Disease-Associated Microglia (DAM) Signature Scoring

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 — DAM signature scoring
# Based on Keren-Shaul et al. Cell 2017 and
# Mathys et al. Nature 2019 DAM gene sets.
# ─────────────────────────────────────────────
DAM_SIG = [
    # DAM stage 1 (TREM2-independent)
    'TYROBP', 'CTSD', 'APOE', 'B2M', 'FTH1',
    # DAM stage 2 (TREM2-dependent)
    'TREM2', 'AXL', 'CST7', 'LPL', 'GPNMB',
    'CD9', 'ITGAX', 'CLEC7A', 'SPP1', 'LGALS3',
    # Lipid metabolism / phagocytosis
    'LIPA', 'HEXA', 'HEXB', 'GRN', 'CTSB',
    # AD-specific upregulation
    'CCL4', 'CCL3', 'CXCL16',
    # Complement
    'C1QA', 'C1QB', 'C1QC',
    # Downregulated in DAM (homeostatic markers)
    # Not included — score only upregulated DAM genes
]

sig_present = [g for g in DAM_SIG if g in adata.raw.var_names]
print(f'DAM signature genes present: {len(sig_present)}/{len(DAM_SIG)}')
missing = [g for g in DAM_SIG if g not in adata.raw.var_names]
if missing: print(f'Missing: {missing}')

sc.tl.score_genes(adata, gene_list=sig_present,
                  score_name='dam_score', use_raw=True)

score_by_cluster = adata.obs.groupby('leiden')['dam_score'].mean()\
                        .sort_values(ascending=False)
print('\nDAM score by cluster (top 10):')
print(score_by_cluster.head(10).round(3))

sc.pl.umap(adata, color='dam_score', color_map='Reds',
           title='Disease-associated microglia (DAM) signature score')

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — Select top DAM clusters
# ─────────────────────────────────────────────
TOP_N_CLUSTERS = 3
top_clusters = score_by_cluster.head(TOP_N_CLUSTERS).index.tolist()
print(f'Top {TOP_N_CLUSTERS} DAM clusters: {top_clusters}')
for c in top_clusters:
    n = (adata.obs['leiden'] == c).sum()
    print(f'  Cluster {c}: {n:,} cells, score {score_by_cluster[c]:.3f}')

adata.obs['is_dam_cluster'] = adata.obs['leiden'].isin(top_clusters).astype(str)
sc.pl.umap(adata, color='is_dam_cluster',
           title=f'Top {TOP_N_CLUSTERS} DAM clusters')

## Phase 5 — Differential Expression

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — Wilcoxon DE
# ─────────────────────────────────────────────
if os.path.exists('ad_DE_genes.csv'):
    de_result = pd.read_csv('ad_DE_genes.csv')
    print(f'Loaded DE genes: {len(de_result)}')
else:
    sc.tl.rank_genes_groups(
        adata, groupby='is_dam_cluster',
        groups=['True'], reference='False',
        method='wilcoxon', use_raw=True
    )
    de_result = sc.get.rank_genes_groups_df(
        adata, group='True', pval_cutoff=0.05, log2fc_min=0.5
    )
    de_result.to_csv('ad_DE_genes.csv', index=False)
    print(f'DE genes (p<0.05, log2FC>0.5): {len(de_result)}')

print(de_result.head(20).to_string())

In [ ]:
# ─────────────────────────────────────────────
# CELL 15 — Volcano plot
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
de_plot = de_result.copy()
de_plot['-log10(padj)'] = -np.log10(de_plot['pvals_adj'].clip(1e-300))
de_plot['color'] = 'gray'
de_plot.loc[(de_plot['pvals_adj'] < 0.05) & (de_plot['logfoldchanges'] > 0.5), 'color'] = '#E24B4A'
de_plot.loc[(de_plot['pvals_adj'] < 0.05) & (de_plot['logfoldchanges'] < -0.5), 'color'] = '#378ADD'
ax.scatter(de_plot['logfoldchanges'], de_plot['-log10(padj)'],
           c=de_plot['color'], alpha=0.5, s=8)
dam_labels = ['TREM2', 'TYROBP', 'AXL', 'CST7', 'LPL', 'GPNMB', 'SPP1', 'APOE']
for _, row in de_plot[de_plot['names'].isin(dam_labels)].iterrows():
    ax.annotate(row['names'], (row['logfoldchanges'], row['-log10(padj)']), fontsize=8)
ax.axhline(-np.log10(0.05), color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('log2 fold change')
ax.set_ylabel('-log10(adj p-value)')
ax.set_title('DE genes: DAM clusters vs rest (AD prefrontal cortex)')
plt.tight_layout()
plt.savefig('ad_volcano.png', dpi=150)
plt.show()

## Phase 6 — ChEMBL Drug Screen

In [ ]:
# ─────────────────────────────────────────────
# CELL 16 — Map DE genes to ChEMBL targets
# DAM-relevant filter: TREM2 pathway, microglial
# kinases, complement, lipid metabolism kinases
# ─────────────────────────────────────────────
TOP_DE_GENES = 50
target_client   = new_client.target
activity_client = new_client.activity

top_de = de_result[
    (de_result['pvals_adj'] < 0.05) &
    (de_result['logfoldchanges'] > 0.5)
].sort_values('logfoldchanges', ascending=False).head(TOP_DE_GENES)

target_genes = top_de['names'].tolist()
print(f'Target genes: {len(target_genes)}')
print(target_genes[:20])

TARGETS_CP = 'chembl_targets_checkpoint.json'
chembl_targets = load_checkpoint(TARGETS_CP)
remaining = [g for g in target_genes if g not in chembl_targets]

for gene in tqdm(remaining, desc='ChEMBL target lookup'):
    try:
        results = target_client.search(gene).filter(
            organism='Homo sapiens', target_type='SINGLE PROTEIN'
        ).only(['target_chembl_id', 'pref_name', 'target_type'])
        hits = list(results)
        chembl_targets[gene] = hits[0]['target_chembl_id'] if hits else None
    except:
        chembl_targets[gene] = None
    time.sleep(0.2)

save_checkpoint(chembl_targets, TARGETS_CP)

# DAM-relevant kinase / receptor filter
DAM_TARGETS = {
    # TREM2 signaling
    'TREM2', 'TYROBP', 'SYK', 'BTK', 'PIK3CG', 'PIK3CD',
    # TAM receptor kinases (AXL, MERTK, TYRO3)
    'AXL', 'MERTK', 'TYRO3',
    # CSF1R (microglial survival kinase)
    'CSF1R',
    # Complement receptors
    'ITGAM', 'ITGB2', 'C1QA', 'C1QB',
    # Lipid / lysosomal
    'GBA', 'LIPA', 'CTSD', 'CTSB',
    # Chemokine receptors
    'CCR2', 'CX3CR1', 'CXCR4',
    # General neuroinflammation kinases
    'MAPK1', 'MAPK3', 'MAPK14', 'MAP2K1',
    'JAK1', 'JAK2', 'STAT3',
    'RIPK1', 'RIPK3',
    # SPP1 / CD44 axis
    'SPP1', 'CD44',
    *sig_present
}
mapped = {k: v for k, v in chembl_targets.items()
          if v and k in target_genes and k in DAM_TARGETS}
print(f'After DAM filter: {len(mapped)} targets')
for g, t in mapped.items(): print(f'  {g}: {t}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 17 — Retrieve ChEMBL inhibitors
# ─────────────────────────────────────────────
CHEMBL_CP = 'chembl_compounds_checkpoint.json'
compound_checkpoint = load_checkpoint(CHEMBL_CP)
all_compounds = []

for gene, target_id in tqdm(mapped.items(), desc='ChEMBL activity'):
    if target_id in compound_checkpoint:
        recs = [dict(r, gene_target=gene) for r in compound_checkpoint[target_id]]
        all_compounds.extend(recs)
        continue
    try:
        results = activity_client.filter(
            target_chembl_id=target_id,
            standard_type__in=['IC50', 'Ki', 'Kd'],
            standard_relation__in=['=', '<', '<='],
            assay_type='B'
        ).only(['molecule_chembl_id', 'canonical_smiles', 'pchembl_value',
                'molecule_pref_name', 'target_chembl_id'])
        hits = list(results)
        for h in hits: h['gene_target'] = gene
        compound_checkpoint[target_id] = hits
        all_compounds.extend(hits)
    except:
        compound_checkpoint[target_id] = []
    time.sleep(0.3)

save_checkpoint(compound_checkpoint, CHEMBL_CP)

df_compounds = pd.DataFrame(all_compounds).drop_duplicates()
df_compounds['pchembl_value'] = pd.to_numeric(
    df_compounds['pchembl_value'], errors='coerce')
df_compounds = df_compounds.dropna(subset=['pchembl_value', 'canonical_smiles'])
df_compounds = df_compounds[df_compounds['pchembl_value'] >= 5.0]
df_compounds = df_compounds.sort_values('pchembl_value', ascending=False)
df_compounds = df_compounds.drop_duplicates(
    subset=['molecule_chembl_id', 'gene_target'], keep='first')
print(f'Compound-target pairs: {len(df_compounds)}')
print(f'Unique compounds: {df_compounds["molecule_chembl_id"].nunique()}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 18 — Priority scoring
# ─────────────────────────────────────────────
compound_summary = df_compounds.groupby('molecule_chembl_id').agg(
    molecule_pref_name=('molecule_pref_name', 'first'),
    canonical_smiles=('canonical_smiles', 'first'),
    n_targets=('gene_target', 'nunique'),
    target_genes=('gene_target', lambda x: ', '.join(sorted(set(x)))),
    max_pchembl=('pchembl_value', 'max'),
    sum_pchembl=('pchembl_value', 'sum'),
).reset_index()

compound_summary['priority_score'] = (
    compound_summary['sum_pchembl'] * compound_summary['n_targets']
).round(1)
compound_summary = compound_summary.sort_values('priority_score', ascending=False)

print(f'Unique compounds: {len(compound_summary)}')
print('\nTop 20:')
print(compound_summary[[
    'molecule_chembl_id', 'molecule_pref_name', 'n_targets',
    'max_pchembl', 'priority_score', 'target_genes'
]].head(20).to_string())

## Phase 7 — USPTO Patent Novelty Check

In [ ]:
# ─────────────────────────────────────────────
# CELL 19 — USPTO novelty check (checkpoint/resume)
# Queries: Alzheimer, microglia, DAM neuroinflammation
# ─────────────────────────────────────────────
USPTO_API  = 'https://developer.uspto.gov/ibd-api/v1/patent/publication'
NOVELTY_CP = 'novelty_checkpoint.json'

def check_uspto_novelty(compound_name, chembl_id):
    if not compound_name or str(compound_name) in ['nan', 'None', '']:
        compound_name = chembl_id
    queries = [
        f'{compound_name} Alzheimer',
        f'{compound_name} microglia neuroinflammation',
        f'{compound_name} TREM2 DAM dementia',
    ]
    total_hits = 0
    for q in queries:
        try:
            resp = requests.get(USPTO_API, params={
                'searchText': q, 'start': 0, 'rows': 5,
                'sources': 'US_PGPUB,USPAT'
            }, timeout=15)
            if resp.status_code == 200:
                total_hits += resp.json().get('response', {}).get('numFound', 0)
            elif resp.status_code == 429:
                print('  Rate limited — sleeping 60s'); time.sleep(60)
        except: pass
        time.sleep(0.3)
    return {
        'molecule_chembl_id': chembl_id,
        'patent_hits': total_hits,
        'novel_flag': 'NOVEL_ALL' if total_hits == 0 else 'PRIOR_ART'
    }

novelty_checkpoint = load_checkpoint(NOVELTY_CP)
already_checked = set(novelty_checkpoint.keys())
remaining_novelty = compound_summary[
    ~compound_summary['molecule_chembl_id'].isin(already_checked)]
print(f'Compounds to check: {len(remaining_novelty)} / {len(compound_summary)}')

batch_count = 0
for i, row in tqdm(remaining_novelty.iterrows(),
                   total=len(remaining_novelty), desc='USPTO'):
    cid = row['molecule_chembl_id']
    novelty_checkpoint[cid] = check_uspto_novelty(
        row.get('molecule_pref_name', ''), cid)
    batch_count += 1
    if batch_count % BATCH_SIZE == 0:
        save_checkpoint(novelty_checkpoint, NOVELTY_CP)
        print(f'  Saved at {batch_count}')

save_checkpoint(novelty_checkpoint, NOVELTY_CP)

df_novelty = pd.DataFrame(list(novelty_checkpoint.values()))
for col in ['patent_hits', 'novel_flag']:
    if col in compound_summary.columns:
        compound_summary = compound_summary.drop(columns=[col])
compound_summary = compound_summary.merge(
    df_novelty[['molecule_chembl_id', 'patent_hits', 'novel_flag']],
    on='molecule_chembl_id', how='left'
)
compound_summary['novel_flag'] = compound_summary['novel_flag'].fillna('UNCHECKED')
print(f"NOVEL_ALL: {(compound_summary['novel_flag'] == 'NOVEL_ALL').sum()} / {len(compound_summary)}")

## Phase 8 — Final Ranking and Export

In [ ]:
# ─────────────────────────────────────────────
# CELL 20 — Final scoring and top 10
# ─────────────────────────────────────────────
compound_summary['novel_mult'] = compound_summary['novel_flag'].map(
    {'NOVEL_ALL': 2.0, 'PRIOR_ART': 1.0, 'UNCHECKED': 1.0})
compound_summary['final_score'] = (
    compound_summary['priority_score'] * compound_summary['novel_mult']
).round(1)
compound_summary = compound_summary.sort_values('final_score', ascending=False)

df_novel = compound_summary[compound_summary['novel_flag'] == 'NOVEL_ALL'].copy()

cols = ['molecule_chembl_id', 'molecule_pref_name', 'n_targets',
        'max_pchembl', 'novel_flag', 'final_score', 'target_genes']
print(f'NOVEL_ALL: {len(df_novel)}')
print('\nTop 10 NOVEL_ALL:')
print(df_novel[cols].head(10).to_string())

In [ ]:
# ─────────────────────────────────────────────
# CELL 21 — Bar chart
# ─────────────────────────────────────────────
top10 = df_novel.head(10).copy()
top10['label'] = top10.apply(
    lambda r: r['molecule_pref_name']
    if (r['molecule_pref_name'] is not None and
        str(r['molecule_pref_name']) not in ['nan', 'None', ''])
    else r['molecule_chembl_id'], axis=1
)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['label'][::-1], top10['final_score'][::-1],
               color='#8E44AD', edgecolor='#4A235A', alpha=0.8)
ax.set_xlabel('Priority Score')
ax.set_title('Top 10 NOVEL_ALL Alzheimer\'s DAM Candidates')
for bar, score in zip(bars, top10['final_score'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{score:.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('ad_top10.png', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 22 — Export results
# ─────────────────────────────────────────────
compound_summary.to_csv('ad_all_candidates.csv', index=False)
df_novel.to_csv('ad_novel_candidates.csv', index=False)
df_novel.head(10).to_csv('ad_top10_patent.csv', index=False)

print('Files saved:')
for f in ['ad_all_candidates.csv', 'ad_novel_candidates.csv',
          'ad_top10_patent.csv', 'ad_DE_genes.csv',
          'ad_volcano.png', 'ad_top10.png',
          'ad_raw.h5ad', 'ad_scvi.h5ad', 'ad_clustered.h5ad']:
    print(f'  {f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 23 — Summary report
# ─────────────────────────────────────────────
top1 = df_novel.iloc[0] if len(df_novel) > 0 else None

print('=' * 60)
print("ALZHEIMER'S DAM SCREEN — SUMMARY REPORT")
print('Ritschel Research')
print('=' * 60)
print(f'Dataset:              GSE138852')
print(f'Cells after QC:       {adata.n_obs:,}')
print(f'Clusters (res 0.5):   {adata.obs["leiden"].nunique()}')
print(f'Top DAM clusters:     {top_clusters}')
print(f'DE genes:             {len(de_result)}')
print(f'ChEMBL targets:       {len(mapped)}')
print(f'Unique compounds:     {len(compound_summary)}')
print(f'NOVEL_ALL:            {len(df_novel)}')
print()
if top1 is not None:
    name = top1.get('molecule_pref_name', top1['molecule_chembl_id'])
    if str(name) in ['nan', 'None', '']: name = top1['molecule_chembl_id']
    print(f'TOP CANDIDATE: {name}')
    print(f'  ChEMBL ID:    {top1["molecule_chembl_id"]}')
    print(f'  Final Score:  {top1["final_score"]}')
    print(f'  pChEMBL:      {top1["max_pchembl"]:.2f}')
    print(f'  Target:       {top1["target_genes"]}')
    print(f'  Patent:       {top1["novel_flag"]}')
print()
print('Next steps:')
print('  1. Review top 10 at ebi.ac.uk/chembl/')
print('  2. Manual Google Patents check (InChI key)')
print('  3. Draft provisional patent specification')
print('  4. Archive on Zenodo + git push to github.com/glenritschel/alzheimers-scrna')